# Strategy Fly

A **simulation** of the FlyWire fruit fly connectome (Shiu et al. 2024 model) plays
**Open Doctrines** through the game's benchmark agent door, under the same action
budget as the game's trained AI. It is not a living fly, and the way game events
reach its taste neurons and its descending neurons reach the game is our invention,
fixed in advance in `PREREGISTRATION.md`.

This notebook stops at the first milestone: **the fly plays a full seat.**
Run all cells in order. Runtime: CPU is enough.

In [ ]:
!mkdir -p /content/strategy_fly_pkg/strategy_fly /content/strategy_fly_pkg/patches /content/results; nproc; free -g | head -2; python3 --version

In [ ]:
%%writefile /content/strategy_fly_pkg/strategy_fly/__init__.py
"""Strategy Fly: a simulated fruit fly brain plays Open Doctrines.

It is a leaky integrate-and-fire simulation of the FlyWire connectome
(Shiu et al. 2024), not a living fly. See PREREGISTRATION.md for every
choice that connects the brain to the game, fixed before any real run.
"""


In [ ]:
%%writefile /content/strategy_fly_pkg/strategy_fly/protocol.py
"""The --bench-agent pipe protocol spoken by OpenDoctrinesServer.

Server side: Game::runBenchAgent in src/Game_AITrain.cpp of Open Doctrines.
Every turn the server prints the position, the legal actions of four menus and
the per-module budget, then `[AGENT] waiting`. It then opens the FIFO and reads
ONE line of comma-separated `module:action` tokens, or `quit`. An empty line is
a turn in which the player does nothing.
"""
import errno
import os
import re
import time

MODULES = "epwn"
MENU_LETTER = {"economy": "e", "politics": "p", "war": "w", "navy": "n"}
DEFAULT_BUDGET = {"e": 8, "p": 3, "w": 8, "n": 8}

_TURN = re.compile(r"^\[AGENT\] ===== turn (\d+)/(\d+)\s+(.*) \(([A-Z0-9]+)\) =====")
_LAND = re.compile(r"^\[AGENT\] land (\d+)/(\d+) \(([\d.]+)% of the world\)\s+army (-?\d+)\s+treasury (-?[\d.]+)")
_INCOME = re.compile(r"^\[AGENT\] gross (-?[\d.]+) net (-?[\d.]+)")
_WARS = re.compile(r"^\[AGENT\] at war with: (.*)$")
_MENU = re.compile(r"^\[AGENT\] (economy|politics|war|navy)\s+(.*)$")
_ACTION = re.compile(r"([epwn]):(\d+) (.+?)(?=\s{2}[epwn]:\d|\s*$)")
_BUDGET = re.compile(r"^\[AGENT\] budget e:(\d+) p:(\d+) w:(\d+) n:(\d+)")
_BENCH = re.compile(r"^\[BENCH\] seat (\S+)\s+score (-?[\d.]+)")


def new_state():
    return {"turn": 0, "turns": 0, "country": "", "iso": "", "share": 0.0,
            "mine": 0, "owned": 0, "army": 0, "treasury": 0.0,
            "gross": 0.0, "net": 0.0, "war": [],
            "legal": {m: [] for m in MODULES}, "budget": dict(DEFAULT_BUDGET),
            "bench_score": None}


def parse_line(line, state):
    """Update `state` from one server line. Returns an event name or None."""
    m = _TURN.match(line)
    if m:
        state.update(turn=int(m[1]), turns=int(m[2]), country=m[3], iso=m[4],
                     legal={k: [] for k in MODULES}, war=[])
        return "turn"
    m = _LAND.match(line)
    if m:
        state.update(mine=int(m[1]), owned=int(m[2]), share=float(m[3]),
                     army=int(m[4]), treasury=float(m[5]))
        return None
    m = _INCOME.match(line)
    if m:
        state.update(gross=float(m[1]), net=float(m[2]))
        return None
    m = _WARS.match(line)
    if m:
        rest = m[1].strip()
        state["war"] = [] if rest == "(nobody)" else rest.split()
        return None
    m = _MENU.match(line)
    if m:
        state["legal"][MENU_LETTER[m[1]]] = [(int(a), n.strip()) for _, a, n in _ACTION.findall(m[2])]
        return None
    m = _BUDGET.match(line)
    if m:
        state["budget"] = dict(zip(MODULES, (int(m[1]), int(m[2]), int(m[3]), int(m[4]))))
        return None
    m = _BENCH.match(line)
    if m:
        state["bench_score"] = float(m[2])
        return "bench"
    if line == "[AGENT] waiting":
        return "waiting"
    if line.startswith("[AGENT] did "):
        return "did"
    if line.startswith("[AGENT] REFUSED"):
        return "refused"
    if line.startswith("[AGENT] OVER BUDGET"):
        return "over_budget"
    if line.startswith("[AGENT] SKIPPED"):
        return "skipped"
    if line.startswith("[AGENT] stopped early"):
        return "stopped"
    return None


def send_line(fifo, line, proc, timeout=60.0):
    """Write one line to the FIFO without ever hanging on a dead server.

    The server prints `waiting` BEFORE it opens the FIFO for reading, so the
    first attempt can find no reader (ENXIO). Retry until it appears, and give
    up if the process has exited.
    """
    deadline = time.time() + timeout
    while True:
        try:
            fd = os.open(fifo, os.O_WRONLY | os.O_NONBLOCK)
            break
        except OSError as e:
            if e.errno != errno.ENXIO:
                raise
            if proc.poll() is not None:
                raise RuntimeError("server exited before reading the turn's choices")
            if time.time() > deadline:
                raise TimeoutError("server never opened the FIFO for reading")
            time.sleep(0.01)
    try:
        os.write(fd, (line + "\n").encode())
    finally:
        os.close(fd)


In [ ]:
%%writefile /content/strategy_fly_pkg/strategy_fly/driver.py
"""Plays one benchmark seat through OpenDoctrinesServer --bench-agent."""
import json
import os
import queue
import subprocess
import threading
import time

from . import protocol


def run_seat(binary, data_dir, seat, seed, player, *, label, turns=120,
             log_dir="results", turn_timeout=3600.0, world_seed=None):
    """Run one seat to the end with `player(state) -> list of tokens`.

    OD_WORLD_SEED is pinned: without it the world seed is drawn from entropy
    and every country's political compass is jittered before the agent door
    applies its own seed (Game::chooseWorldSeed, jitterStartingPolitics), so
    two runs of one seed would not be the same world.
    """
    os.makedirs(log_dir, exist_ok=True)
    tag = f"{label}__{seat.replace(':', '_')}__{seed}"
    fifo = f"/tmp/strategy_fly_{os.getpid()}_{tag}.fifo"
    if os.path.exists(fifo):
        os.unlink(fifo)
    os.mkfifo(fifo)

    env = dict(os.environ)
    env["OD_WORLD_SEED"] = str(world_seed if world_seed is not None else seed)
    cmd = [binary, "--bench-agent", seat, fifo, "--until", str(turns),
           "--seed", str(seed), "--data", data_dir]
    log_path = os.path.join(log_dir, tag + ".log")
    turns_path = os.path.join(log_dir, tag + ".turns.jsonl")
    counts = {"did": 0, "refused": 0, "over_budget": 0, "skipped": 0}

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=env)
    lines = queue.Queue()
    log = open(log_path, "w")

    def pump():
        # Drained continuously: a full stdout pipe would block the server mid-turn.
        for ln in proc.stdout:
            log.write(ln)
            lines.put(ln.rstrip("\n"))
        lines.put(None)

    threading.Thread(target=pump, daemon=True).start()
    state = protocol.new_state()
    started = time.time()
    try:
        with open(turns_path, "w") as tl:
            while True:
                try:
                    ln = lines.get(timeout=turn_timeout)
                except queue.Empty:
                    raise TimeoutError(f"{tag}: no output for {turn_timeout}s")
                if ln is None:
                    break
                ev = protocol.parse_line(ln, state)
                if ev in counts:
                    counts[ev] += 1
                if ev == "waiting":
                    t0 = time.time()
                    tokens = list(player(state))
                    think = time.time() - t0
                    protocol.send_line(fifo, ", ".join(tokens), proc)
                    rec = {k: state[k] for k in ("turn", "share", "army", "treasury", "gross", "net", "war", "budget")}
                    rec.update(tokens=tokens, think_s=round(think, 3),
                               info=getattr(player, "last_info", None))
                    tl.write(json.dumps(rec) + "\n")
                    tl.flush()
        proc.wait(timeout=120)
    finally:
        if proc.poll() is None:
            proc.kill()
        log.close()
        if os.path.exists(fifo):
            os.unlink(fifo)
    return {"label": label, "seat": seat, "seed": seed, "turns": turns,
            "score": state["bench_score"], "returncode": proc.returncode,
            "counts": counts, "wall_s": round(time.time() - started, 1),
            "log": log_path, "turns_log": turns_path}


In [ ]:
%%writefile /content/strategy_fly_pkg/strategy_fly/brain.py
"""The whole-brain model, built once and run in short decision windows.

Neuron and synapse equations and every constant are copied unchanged from
model.py of philshiu/Drosophila_brain_model (MIT), which accompanies Shiu et
al., "A Drosophila computational brain model reveals sensorimotor processing",
Nature 2024. Two things differ, both for speed and neither for dynamics:

- The network is built ONCE. model.py rebuilds all 15 million synapses for
  every trial, which a game of 120 turns cannot afford.
- Stimulation uses a PoissonGroup per sensory channel instead of one
  PoissonInput per neuron. PoissonInput fixes its rate when it is created;
  a PoissonGroup's `rates` can change every turn. Each input spike adds
  w_syn * f_poi to the target's membrane, exactly as PoissonInput did, and
  the targets have no refractory period, as in model.py.
"""
import resource
import sys
import time
from textwrap import dedent

import numpy as np
import pandas as pd
from brian2 import (Hz, NeuronGroup, Network, PoissonGroup, SpikeMonitor,
                    Synapses, mV, ms, prefs)
from brian2 import seed as brian_seed

PARAMS = {
    "v_0": -52 * mV, "v_rst": -52 * mV, "v_th": -45 * mV,
    "t_mbr": 20 * ms, "tau": 5 * ms, "t_rfc": 2.2 * ms, "t_dly": 1.8 * ms,
    "w_syn": 0.275 * mV, "f_poi": 250,
}
EQS = dedent("""
    dv/dt = (v_0 - v + g) / t_mbr : volt (unless refractory)
    dg/dt = -g / tau               : volt (unless refractory)
    rfc                            : second
    """)
THRESHOLD = "v > v_th"
RESET = "v = v_rst; w = 0; g = 0 * mV"


def peak_rss_mb():
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return r / 1024 if sys.platform.startswith("linux") else r / 1048576


class FlyBrain:
    def __init__(self, completeness_csv, connectivity_parquet, *,
                 shuffle_seed=None, codegen="cython"):
        t0 = time.time()
        prefs.codegen.target = codegen
        comp = pd.read_csv(completeness_csv, index_col=0)
        self.flyids = [str(x) for x in comp.index]
        self.index = {f: i for i, f in enumerate(self.flyids)}
        self.n = len(self.flyids)
        del comp

        con = pd.read_parquet(connectivity_parquet, columns=[
            "Presynaptic_Index", "Postsynaptic_Index", "Excitatory x Connectivity"])
        pre = con["Presynaptic_Index"].to_numpy(np.int32)
        post = con["Postsynaptic_Index"].to_numpy(np.int32)
        weights = con["Excitatory x Connectivity"].to_numpy(np.float64)
        self.n_synapses = len(pre)
        del con

        # THE SHUFFLED CONTROL. Permuting only the postsynaptic column keeps
        # every neuron's out-degree, in-degree and outgoing signs and weights,
        # and scrambles who talks to whom.
        self.shuffle_seed = shuffle_seed
        if shuffle_seed is not None:
            post = np.random.default_rng(shuffle_seed).permutation(post)

        self.neu = NeuronGroup(self.n, model=EQS, method="linear", threshold=THRESHOLD,
                               reset=RESET, refractory="rfc", name="neurons",
                               namespace=dict(PARAMS))
        self.neu.v = PARAMS["v_0"]
        self.neu.g = 0 * mV
        self.neu.rfc = PARAMS["t_rfc"]
        self.syn = Synapses(self.neu, self.neu, "w : volt", on_pre="g += w",
                            delay=PARAMS["t_dly"], name="synapses")
        self.syn.connect(i=pre, j=post)
        self.syn.w = weights * PARAMS["w_syn"]
        del pre, post, weights

        self.mon = SpikeMonitor(self.neu, record=False, name="spikes")
        self.net = Network(self.neu, self.syn, self.mon)
        self.channels = {}
        self.build_seconds = time.time() - t0

    def add_channel(self, name, flyids):
        """A sensory population driven by one Poisson rate. Returns its size."""
        idx = np.array([self.index[f] for f in flyids if f in self.index], dtype=np.int32)
        pg = PoissonGroup(len(idx), rates=0 * Hz, name=f"in_{name}")
        syn = Synapses(pg, self.neu, on_pre="v += w_in", name=f"in_{name}_syn",
                       namespace={"w_in": PARAMS["w_syn"] * PARAMS["f_poi"]})
        syn.connect(i=np.arange(len(idx)), j=idx)
        self.neu.rfc[idx] = 0 * ms
        self.net.add(pg, syn)
        self.channels[name] = (pg, idx)
        return len(idx)

    def run_window(self, rates_hz, window_ms, rng_seed, flush_ms=2.0):
        """Spike counts per neuron over one decision window.

        Inputs off and a short run first, longer than the 1.8 ms synaptic
        delay, so spikes still in flight from the previous turn land before
        the membrane is reset and counting starts.
        """
        brian_seed(int(rng_seed) & 0x7FFFFFFF)
        for pg, _ in self.channels.values():
            pg.rates = 0 * Hz
        if flush_ms > 0:
            self.net.run(flush_ms * ms)
        self.neu.v = PARAMS["v_0"]
        self.neu.g = 0 * mV
        for name, (pg, _) in self.channels.items():
            pg.rates = float(rates_hz.get(name, 0.0)) * Hz
        before = np.array(self.mon.count[:], dtype=np.int64)
        self.net.run(window_ms * ms)
        return np.array(self.mon.count[:], dtype=np.int64) - before


In [ ]:
%%writefile /content/strategy_fly_pkg/strategy_fly/encode.py
"""Game state -> stimulation rates of real sensory neuron populations.

Fixed before any real run; see PREREGISTRATION.md. Rates stay inside the
ranges Shiu et al. used (roughly 10-200 Hz).
"""
REFS = {"land_pp": 0.5, "reserve_turns": 12.0, "wars": 3.0}
MAX_HZ = 200.0
FLOOR_HZ = 10.0
CHANNELS = ("sugar", "bitter", "water", "jon")


class Encoder:
    def __init__(self):
        self.prev = None

    def signals(self, s):
        share = float(s.get("share", 0.0))
        gross = float(s.get("gross", 0.0))
        net = float(s.get("net", 0.0))
        treasury = float(s.get("treasury", 0.0))
        wars = list(s.get("war", []))
        if self.prev is None:
            d_land, new_wars = 0.0, 0
        else:
            d_land = share - self.prev["share"]
            new_wars = len(set(wars) - set(self.prev["war"]))
        ratio = net / gross if gross > 1e-9 else 0.0
        sig = {
            "sugar": max(0.0, d_land) / REFS["land_pp"] + max(0.0, ratio),
            "bitter": max(0.0, -d_land) / REFS["land_pp"] + max(0.0, -ratio) + 0.5 * new_wars,
            "water": treasury / (REFS["reserve_turns"] * gross) if gross > 1e-9 else 0.0,
            "jon": len(wars) / REFS["wars"],
        }
        self.prev = {"share": share, "war": wars}
        return sig

    def rates(self, s):
        out = {}
        for k, v in self.signals(s).items():
            r = MAX_HZ * min(1.0, max(0.0, v))
            out[k] = r if r >= FLOOR_HZ else 0.0
        return out


In [ ]:
%%writefile /content/strategy_fly_pkg/strategy_fly/decode.py
"""Descending-neuron spikes -> actions, under the model's own budget.

The 1,303 descending neurons (FlyWire super_class "descending") are the
brain's output to the body. They carry no annotation that says "declare war",
so any assignment of neurons to actions is arbitrary. This one is at least
not chosen by hand: whole cell types (so left and right homologues move
together) are dealt into 39 balanced groups by a committed seed, and each
group is one action. The shuffled-connectome control uses the same groups.
"""
import csv

import numpy as np

MENU_SIZES = {"e": 12, "p": 12, "w": 8, "n": 7}
MODULES = "epwn"


def descending_units(annotations_tsv, flyid_to_index):
    """Lists of brain indices, one per descending cell type."""
    units = {}
    with open(annotations_tsv, newline="") as f:
        for row in csv.DictReader(f, delimiter="\t"):
            if row["super_class"] != "descending":
                continue
            i = flyid_to_index.get(row["root_id"])
            if i is None:
                continue
            key = row["cell_type"] or row["hemibrain_type"] or ("root:" + row["root_id"])
            units.setdefault(key, []).append(i)
    return [sorted(units[k]) for k in sorted(units)]


def make_groups(units, seed):
    """Deal units into one balanced group per action. Deterministic in `seed`."""
    rng = np.random.default_rng(seed)
    n_groups = sum(MENU_SIZES.values())
    order = list(rng.permutation(len(units)))
    order.sort(key=lambda u: -len(units[u]))          # stable: seed breaks size ties
    bins = [[] for _ in range(n_groups)]
    totals = [0] * n_groups
    for u in order:
        low = min(totals)
        candidates = [b for b in range(n_groups) if totals[b] == low]
        b = candidates[int(rng.integers(len(candidates)))]
        bins[b].extend(units[u])
        totals[b] += len(units[u])
    slots = list(rng.permutation(n_groups))
    groups, k = {}, 0
    for m in MODULES:
        groups[m] = [np.array(sorted(bins[slots[k + a]]), dtype=np.int64) for a in range(MENU_SIZES[m])]
        k += MENU_SIZES[m]
    return groups


def choose(counts, groups, state, rng):
    """Up to the module's budget of the most active legal actions, per menu.

    Score = mean spikes per neuron in the action's group. Actions with no
    spikes are not taken. A menu where nothing fired holds (action 0). The
    server ends a module when it takes action 0, so nothing ranked after it
    is sent. Ties are broken by `rng`, seeded per turn.
    """
    tokens, info = [], {}
    for m in MODULES:
        legal = [a for a, _ in state["legal"][m]]
        if not legal:
            continue
        budget = int(state["budget"][m])
        scores = {a: float(counts[groups[m][a]].sum()) / len(groups[m][a]) for a in legal}
        order = sorted(legal, key=lambda a: (-scores[a], rng.random()))
        picks = [a for a in order if scores[a] > 0][:budget]
        if not picks:
            picks = [0] if 0 in legal else []
        if 0 in picks:
            picks = picks[:picks.index(0) + 1]
        tokens += [f"{m}:{a}" for a in picks]
        info[m] = {"picks": picks, "best": max(scores.values())}
    return tokens, info


In [ ]:
%%writefile /content/strategy_fly_pkg/strategy_fly/players.py
"""The players: the simulated fly, and the baselines that play the same door."""
import random
import zlib

import numpy as np

from . import decode


def turn_seed(base, iso, turn):
    return zlib.crc32(f"{base}|{iso}|{turn}".encode()) & 0x7FFFFFFF


class FlyPlayer:
    """The brain decides every turn. `blind_rates` gives the input-blind control."""

    def __init__(self, brain, groups, encoder, *, window_ms=200.0, seed=0, blind_rates=None):
        self.brain, self.groups, self.encoder = brain, groups, encoder
        self.window_ms, self.seed, self.blind_rates = window_ms, seed, blind_rates
        self.dn_all = np.unique(np.concatenate([g for m in decode.MODULES for g in groups[m]]))
        self.last_info = None

    def __call__(self, state):
        rates = dict(self.blind_rates) if self.blind_rates is not None else self.encoder.rates(state)
        s = turn_seed(self.seed, state["iso"], state["turn"])
        counts = self.brain.run_window(rates, self.window_ms, s)
        tokens, info = decode.choose(counts, self.groups, state, random.Random(s))
        self.last_info = {"rates": rates,
                          "dn_spikes": int(counts[self.dn_all].sum()),
                          "active_neurons": int((counts > 0).sum()),
                          "picks": {m: v["picks"] for m, v in info.items()}}
        return tokens


class AlwaysHold:
    """Never does anything. Action 0 is legal in every menu."""
    last_info = None

    def __call__(self, state):
        return [f"{m}:0" for m in decode.MODULES if any(a == 0 for a, _ in state["legal"][m])]


class RandomLegal:
    """Uniformly random legal actions, a random number of them up to the budget."""

    def __init__(self, seed=0):
        self.seed = seed
        self.last_info = None

    def __call__(self, state):
        rng = random.Random(turn_seed(self.seed, state["iso"], state["turn"]))
        tokens = []
        for m in decode.MODULES:
            legal = [a for a, _ in state["legal"][m]]
            if not legal:
                continue
            for _ in range(rng.randint(1, int(state["budget"][m]))):
                a = rng.choice(legal)
                tokens.append(f"{m}:{a}")
                if a == 0:
                    break
        return tokens


In [ ]:
%%writefile /content/strategy_fly_pkg/strategy_fly/sensory_ids.json
{
 "_source": "Neuron lists from example.ipynb and figures.ipynb of philshiu/Drosophila_brain_model (MIT); FlyWire root ids. The model's notebooks were written for FlyWire v630; ids absent from v783 are skipped at load time.",
 "sugar": [
  "720575940624963786",
  "720575940630233916",
  "720575940637568838",
  "720575940638202345",
  "720575940617000768",
  "720575940630797113",
  "720575940632889389",
  "720575940621754367",
  "720575940621502051",
  "720575940640649691",
  "720575940639332736",
  "720575940616885538",
  "720575940639198653",
  "720575940620900446",
  "720575940617937543",
  "720575940632425919",
  "720575940633143833",
  "720575940612670570",
  "720575940628853239",
  "720575940629176663",
  "720575940611875570"
 ],
 "bitter": [
  "720575940621778381",
  "720575940602353632",
  "720575940617094208",
  "720575940619197093",
  "720575940626287336",
  "720575940618600651",
  "720575940627692048",
  "720575940630195909",
  "720575940646212996",
  "720575940610483162",
  "720575940645743412",
  "720575940627578156",
  "720575940622298631",
  "720575940621008895",
  "720575940629146711",
  "720575940610259370",
  "720575940610481370",
  "720575940619028208",
  "720575940614281266",
  "720575940613061118",
  "720575940604027168"
 ],
 "water": [
  "720575940612950568",
  "720575940631898285",
  "720575940606002609",
  "720575940612579053",
  "720575940622902535",
  "720575940616177458",
  "720575940660292225",
  "720575940622486922",
  "720575940613786774",
  "720575940629852866",
  "720575940625861168",
  "720575940613996959",
  "720575940617857694",
  "720575940644965399",
  "720575940625203504",
  "720575940630553415",
  "720575940635172191",
  "720575940634796536"
 ],
 "jon": [
  "720575940619341105",
  "720575940630122015",
  "720575940611061526",
  "720575940615848788",
  "720575940628444667",
  "720575940627941431",
  "720575940632449619",
  "720575940650244342",
  "720575940631866508",
  "720575940638681845",
  "720575940628978450",
  "720575940609522461",
  "720575940621442224",
  "720575940602506208",
  "720575940629022149",
  "720575940627109991",
  "720575940630020111",
  "720575940615986459",
  "720575940618684481",
  "720575940620382889",
  "720575940630080071",
  "720575940626565455",
  "720575940630319671",
  "720575940602720940",
  "720575940630564179",
  "720575940637632419",
  "720575940615809349",
  "720575940626042149",
  "720575940637054835",
  "720575940602132509",
  "720575940614188149",
  "720575940616951124",
  "720575940628101126",
  "720575940629055721",
  "720575940616589878",
  "720575940622449388",
  "720575940614427195",
  "720575940625797617",
  "720575940638664437",
  "720575940618467195",
  "720575940621729757",
  "720575940613971485",
  "720575940627585688",
  "720575940629650997",
  "720575940630059847",
  "720575940608742409",
  "720575940614351477",
  "720575940633153375",
  "720575940622937528",
  "720575940604753437",
  "720575940611783464",
  "720575940618599872",
  "720575940609541917",
  "720575940637410869",
  "720575940630070343",
  "720575940621397417",
  "720575940614035485",
  "720575940610018266",
  "720575940626307902",
  "720575940634634606",
  "720575940614060829",
  "720575940624799290",
  "720575940641921421",
  "720575940623298559",
  "720575940625559358",
  "720575940629138959",
  "720575940621625597",
  "720575940625962568",
  "720575940632767383",
  "720575940624915230",
  "720575940606239243",
  "720575940626956777",
  "720575940604973746",
  "720575940622222856",
  "720575940642517284",
  "720575940629719404",
  "720575940616613022",
  "720575940604299454",
  "720575940615473186",
  "720575940622217992",
  "720575940606800341",
  "720575940629267498",
  "720575940637366335",
  "720575940624224408",
  "720575940609543197",
  "720575940633364179",
  "720575940629502009",
  "720575940606431189",
  "720575940625733960",
  "720575940638529525",
  "720575940617524053",
  "720575940628935564",
  "720575940624308355",
  "720575940631170346",
  "720575940627704375",
  "720575940625885512",
  "720575940614929245",
  "720575940647493241",
  "720575940618888368",
  "720575940625087546",
  "720575940606657493",
  "720575940617273560",
  "720575940640591861",
  "720575940639410035",
  "720575940621532413",
  "720575940627523584",
  "720575940621521917",
  "720575940621097398",
  "720575940625915338",
  "720575940606222428",
  "720575940627868471",
  "720575940622179497",
  "720575940608297774",
  "720575940614026269",
  "720575940613012959",
  "720575940628100614",
  "720575940606611401",
  "720575940628649465",
  "720575940610008217",
  "720575940623791152",
  "720575940625571240",
  "720575940634923621",
  "720575940609530653",
  "720575940635968745",
  "720575940625703434",
  "720575940613105311",
  "720575940629386819",
  "720575940623077389",
  "720575940625763015",
  "720575940628359017",
  "720575940630834171",
  "720575940622892988",
  "720575940621289537",
  "720575940641395163",
  "720575940616064546",
  "720575940628978409",
  "720575940652566177",
  "720575940627493096",
  "720575940619085397",
  "720575940635545310",
  "720575940645728803",
  "720575940629141775",
  "720575940626557995",
  "720575940631098338",
  "720575940639904475",
  "720575940635067034"
 ]
}

In [ ]:
%%writefile /content/strategy_fly_pkg/patches/opendoctrines-agent-door.patch
diff --color -ru a/src/Game_AITrain.cpp b/src/Game_AITrain.cpp
--- a/src/Game_AITrain.cpp	2026-09-13 19:34:23.905774763 +0200
+++ b/src/Game_AITrain.cpp	2026-09-13 19:34:23.965911238 +0200
@@ -2549,6 +2549,7 @@
     applyFpsTarget(-1);
     Audio::s_disabled = true;
 
+    m_config.aiDifficulty = 3;   // Strategy Fly: match tools/od_bench.py DIFFICULTY
     startBenchSeat(seatSpec, untilTurn);
     while (m_loadingPhase != LOAD_NONE && m_loadingPhase != LOAD_DONE) {
         if (WindowShouldClose()) return false;
@@ -2646,6 +2647,16 @@
             }
             printf("[AGENT] %-8s %s\n", MODS[mod].label, line.c_str());
         }
+        // ── THE MODEL'S BUDGET, STATED ──
+        //
+        // A difference in score is only a difference in judgement if both
+        // players get the same number of moves. The policy makes up to
+        // agentBudget() picks per module each turn and stops a module when it
+        // picks 0; before this, an agent could send any number of actions and
+        // the comparison measured the budget rather than the player.
+        printf("[AGENT] budget e:%d p:%d w:%d n:%d  (a module also ends when it picks 0)\n",
+               m_ai->agentBudget(cid, 0), m_ai->agentBudget(cid, 1),
+               m_ai->agentBudget(cid, 2), m_ai->agentBudget(cid, 3));
         printf("[AGENT] waiting\n");
         fflush(stdout);
 
@@ -2666,6 +2677,8 @@
         while (!cmds.empty() && (cmds.back() == '\n' || cmds.back() == '\r')) cmds.pop_back();
         if (cmds == "quit") { printf("[AGENT] stopped early at turn %d\n", m_turnNumber); break; }
 
+        int used[4] = {0, 0, 0, 0};
+        bool passed[4] = {false, false, false, false};
         size_t at = 0;
         while (at < cmds.size()) {
             const size_t comma = cmds.find(',', at);
@@ -2684,6 +2697,20 @@
                 printf("[AGENT] REFUSED %s: not legal this turn\n", tok.c_str());
                 continue;
             }
+            // After the legality check, so a refused token costs nothing -- the
+            // policy only ever picks from the legal set, and would not have
+            // spent a move on it either.
+            if (passed[mod]) {
+                printf("[AGENT] SKIPPED %s: this module already picked 0 this turn\n", tok.c_str());
+                continue;
+            }
+            const int budget = m_ai->agentBudget(cid, mod);
+            if (used[mod] >= budget) {
+                printf("[AGENT] OVER BUDGET %s: %d of %d this turn\n", tok.c_str(), used[mod], budget);
+                continue;
+            }
+            ++used[mod];
+            if (act == 0) passed[mod] = true;
             printf("[AGENT] did %s -> %s\n", tok.c_str(),
                    m_ai->agentExec(cid, mod, act).c_str());
             m_ai->agentRefresh();
diff --color -ru a/src/ai/AISystem.h b/src/ai/AISystem.h
--- a/src/ai/AISystem.h	2026-09-13 19:34:23.888212982 +0200
+++ b/src/ai/AISystem.h	2026-09-13 19:34:23.936587130 +0200
@@ -2153,6 +2153,16 @@
             default:           return execNavy(cid, action);
         }
     }
+    // The policy's own per-turn budget for a module, so a hand-played seat gets
+    // exactly as many picks as the model does: politics keeps its rate limit,
+    // the other three get actionsPerModule(). The same loop in takeTurn also
+    // ends a module the moment it picks action 0, and runBenchAgent applies
+    // that rule too -- a budget without it would let an agent pass and then
+    // act again, which the model cannot do.
+    int agentBudget(int cid, int module) const {
+        return module == MOD_POLITICS ? ACTIONS_PER_MODULE_PER_TURN
+                                      : actionsPerModule(cid);
+    }
 private:
 public:
     // Decide + enqueue this country's orders through the same pending-order


## 1. Build the game server (headless, about 10 minutes)

In [ ]:
%%bash
set -e
mkdir -p /content/strategy_fly_pkg/strategy_fly /content/strategy_fly_pkg/patches /content/results
apt-get -qq update
apt-get -qq install -y build-essential cmake ninja-build git git-lfs python3 libasound2-dev libx11-dev libxrandr-dev libxi-dev libgl1-mesa-dev libglu1-mesa-dev libxcursor-dev libxinerama-dev libwayland-dev libxkbcommon-dev > /dev/null
cd /content
[ -d od ] || git clone -q https://github.com/Pr1nted/Open-Doctrines od
cd od
git checkout -q 05881e0
git lfs pull > /dev/null 2>&1 || true
if git apply --reverse --check /content/strategy_fly_pkg/patches/opendoctrines-agent-door.patch 2>/dev/null; then
  echo "patch already applied"
else
  git apply -p1 /content/strategy_fly_pkg/patches/opendoctrines-agent-door.patch && echo "patch applied"
fi
# -include cstdint: Colab builds with GCC 13, which no longer brings <cstdint> in
# through <string> or <algorithm>. Open Doctrines headers that use uint8_t without
# including it built on CI (GCC 11) and stop here. Force-including it fixes the whole
# class at once and changes nothing about what the game does.
cmake -S . -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DCMAKE_CXX_FLAGS="-include cstdint" > /content/results/cmake-configure.log
time cmake --build build --target OpenDoctrinesServer -j"$(nproc)" > /content/results/cmake-build.log
ls -la build/OpenDoctrinesServer

## 2. Check the agent door before any brain touches it

In [ ]:
import sys; sys.path.insert(0, "/content/strategy_fly_pkg")
from strategy_fly.driver import run_seat
from strategy_fly.players import AlwaysHold
BIN, DATA = "/content/od/build/OpenDoctrinesServer", "/content/od/data/"
r = run_seat(BIN, DATA, "1914:SWE:rung", 20260801, AlwaysHold(), label="door-check", turns=3, log_dir="/content/results")
print(r)
import subprocess; print(subprocess.run(["grep", "-m3", "budget\\|BENCH\\|seed", r["log"]], capture_output=True, text=True).stdout)
assert r["returncode"] == 0 and r["score"] is not None, "the door did not complete a 3-turn seat"

## 3. Download the connectome and build the brain

In [ ]:
%%bash
set -e
pip -q install "brian2==2.10.1" pyarrow
mkdir -p /content/fly && cd /content/fly
B=https://raw.githubusercontent.com/philshiu/Drosophila_brain_model/main
[ -f Completeness_783.csv ] || curl -sSLO $B/Completeness_783.csv
[ -f Connectivity_783.parquet ] || curl -sSLO $B/Connectivity_783.parquet
[ -f neuron_annotations.tsv ] || curl -sSL -o neuron_annotations.tsv https://raw.githubusercontent.com/flyconnectome/flywire_annotations/main/supplemental_files/Supplemental_file1_neuron_annotations.tsv
ls -la

In [ ]:
import json, time
from strategy_fly.brain import FlyBrain, peak_rss_mb
from strategy_fly import decode
from strategy_fly.encode import Encoder
from strategy_fly.players import FlyPlayer, RandomLegal

PARTITION_SEED, BRAIN_SEED, WINDOW_MS = 783, 20240922, 200.0   # fixed in PREREGISTRATION.md
brain = FlyBrain("/content/fly/Completeness_783.csv", "/content/fly/Connectivity_783.parquet")
ids = json.load(open("/content/strategy_fly_pkg/strategy_fly/sensory_ids.json"))
sizes = {ch: brain.add_channel(ch, ids[ch]) for ch in ("sugar", "bitter", "water", "jon")}
units = decode.descending_units("/content/fly/neuron_annotations.tsv", brain.index)
groups = decode.make_groups(units, PARTITION_SEED)
print(f"neurons {brain.n}, synapses {brain.n_synapses}, built in {brain.build_seconds:.0f} s, peak RSS {peak_rss_mb():.0f} MB")
print("sensory channels:", sizes, "| descending cell types:", len(units))

## 4. Stage 0 and 1: speed, and does the output respond at all?

In [ ]:
import numpy as np
dn_all = np.unique(np.concatenate([g for m in decode.MODULES for g in groups[m]]))
probes = {"rest": {}, "sugar": {"sugar": 200}, "bitter": {"bitter": 200},
          "water": {"water": 200}, "jon": {"jon": 200},
          "crisis": {"bitter": 200, "jon": 130}}
report = {}
for name, rates in probes.items():
    t0 = time.time()
    c = brain.run_window(rates, WINDOW_MS, rng_seed=1)
    report[name] = {"wall_s": round(time.time() - t0, 1), "active": int((c > 0).sum()),
                    "dn_spikes": int(c[dn_all].sum())}
    print(name, report[name])
responsive = any(v["dn_spikes"] > 0 for k, v in report.items() if k != "rest")
print("descending neurons respond to input:", responsive)
assert responsive, "STOP: the output neurons never fire, so the fly could only ever hold."

## 5. The fly plays a full seat

In [ ]:
fly = FlyPlayer(brain, groups, Encoder(), window_ms=WINDOW_MS, seed=BRAIN_SEED)
result = run_seat(BIN, DATA, "1914:SWE:rung", 20260801, fly, label="fly", turns=120, log_dir="/content/results")
print(result)
assert result["returncode"] == 0 and result["score"] is not None
assert result["counts"]["did"] > 0, "the fly never took an action"
print("The fly played Open Doctrines:", result["counts"]["did"], "actions over 120 turns, final share", result["score"])

In [ ]:
# Context only, same door, same seat and seed. Not the benchmark: see PREREGISTRATION.md.
for label, player in (("hold", AlwaysHold()), ("random", RandomLegal(seed=BRAIN_SEED))):
    print(run_seat(BIN, DATA, "1914:SWE:rung", 20260801, player, label=label, turns=120, log_dir="/content/results"))

In [ ]:
!cd /content && zip -qr strategy_fly_results.zip results && ls -la strategy_fly_results.zip